# ¿Qué tan mejor puede ser el modelo ganador? — hiperparámetros y feature set por target

## Qué responde este notebook

XGBoost ya ganó en los 3 targets (`13_comparacion_modelos.ipynb`, `14_receiving_tds_modelos_de_conteo.ipynb`),
con una búsqueda pequeña de hiperparámetros (12 combinaciones: `max_depth`/`n_estimators`/
`learning_rate`) y el mismo feature set de 34 columnas compartido entre los 3 targets. Este
notebook no vuelve a competir familias de modelo (Ridge/RandomForest/etc.) — eso ya se decidió —
prueba si el **mismo** XGBoost puede mejorar por 2 caminos distintos, cada uno aislado para saber
cuál realmente aporta:

1. **Hiperparámetros**: una búsqueda mucho más amplia (regularización L1/L2, submuestreo,
   `min_child_weight`, `gamma` — nada de esto se había explorado), con el mismo feature set
   compartido de siempre.
2. **Feature set propio por target**: en vez de las 34 columnas compartidas por los 3 targets
   (que resultaron ser la unión de los top 15 de cada uno, no que las 34 sirvan igual a los 3 —
   varias tienen importancia casi nula o negativa para un target aunque ayuden a otro), usar solo
   el top 15 propio de cada target, con los hiperparámetros que ya ganaron antes.

Y la combinación de ambos, para ver si se refuerzan o no.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
features_compartidas = features.columnas_por_modelo("arbol")
train_mask, val_mask, test_mask = modeling.dividir_temporal(
    wr, range(2016, 2022), [2022, 2023], [2024, 2025]
)
TARGETS = ["receptions", "receiving_yards", "receiving_tds"]

from xgboost import XGBRegressor as _XGBRegressor

_modelo_tds_baseline = _XGBRegressor(
    objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1
)
_modelo_tds_baseline.fit(wr.loc[train_mask, features_compartidas], wr.loc[train_mask, "receiving_tds"])
_val_mae_tds_baseline = modeling.metricas(
    wr.loc[val_mask, "receiving_tds"],
    _modelo_tds_baseline.predict(wr.loc[val_mask, features_compartidas]),
)["mae"]

BASELINE = {
    "receptions": {"val_mae": 1.449, "test_mae": 1.372},
    "receiving_yards": {"val_mae": 22.015, "test_mae": 20.647},
    "receiving_tds": {"val_mae": _val_mae_tds_baseline, "test_mae": 0.2466},
}
HIPERPARAMETROS_BASELINE = {
    "receptions": dict(max_depth=4, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="reg:tweedie", tweedie_variance_power=1.5),
}
PARAMS_FIJOS = {"receiving_tds": {"objective": "reg:tweedie", "tweedie_variance_power": 1.5}}
print(f"features compartidas: {len(features_compartidas)} columnas")
print(f"baseline receiving_tds -- val_mae recalculado: {_val_mae_tds_baseline:.4f}")

features compartidas: 34 columnas
baseline receiving_tds -- val_mae recalculado: 0.2464


## 1. El feature set propio de cada target (top 15, de `11_seleccion_features.ipynb`)

Reusa los top 15 ya calculados y verificados en Fase 4 -- no se recalcula la importancia aquí, se
toma la lista tal cual quedó registrada.

In [2]:
FEATURES_PROPIAS = {
    "receptions": [
        "target_share_last5_avg", "receptions_season_avg", "depth_team", "receptions_last3_avg",
        "receptions_last5_avg", "receptions_career_avg", "targets_season_avg", "targets_last5_avg",
        "draft_pick", "receiving_10_last3_avg", "receiving_epa_last3_avg", "targets_last3_avg",
        "target_share_career_avg", "receiving_20_last3_avg", "target_share_last3_avg",
    ],
    "receiving_yards": [
        "target_share_last5_avg", "depth_team", "wopr_last5_avg", "fantasy_points_ppr_last5_avg",
        "targets_season_avg", "receptions_season_avg", "receiving_yards_season_avg",
        "receiving_first_downs_last5_avg", "wopr_last3_avg", "targets_last5_avg",
        "air_yards_share_season_avg", "receiving_yards_career_avg", "receiving_yards_last3_avg",
        "draft_pick", "receiving_air_yards_season_avg",
    ],
    "receiving_tds": [
        "targets_last5_avg", "air_yards_share_career_avg", "draft_pick", "fantasy_points_ppr_last5_avg",
        "receiving_yards_career_avg", "receiving_first_downs_career_avg", "receiving_10_last3_avg",
        "depth_team", "air_yards_share_last3_avg", "target_share_last3_avg",
        "fantasy_points_ppr_season_avg", "receiving_tds_season_avg", "receiving_16_last5_avg",
        "catch_rate_career_avg", "fantasy_points_ppr_last3_avg",
    ],
}
for target, cols in FEATURES_PROPIAS.items():
    faltantes = [c for c in cols if c not in wr.columns]
    print(f"{target}: {len(cols)} columnas propias, faltantes: {faltantes}")

receptions: 15 columnas propias, faltantes: []
receiving_yards: 15 columnas propias, faltantes: []
receiving_tds: 15 columnas propias, faltantes: []


## 2. Eje 1 — hiperparámetros más amplios, mismo feature set compartido

100 combinaciones aleatorias por target, buscando en profundidad, tasa de aprendizaje,
submuestreo de filas/columnas, `min_child_weight`, y regularización L1/L2/`gamma` — nada de esto
se había explorado antes (la búsqueda original solo variaba profundidad/n_estimators/tasa).
`n_estimators` no se prueba como valor fijo: se usa *early stopping* sobre el conjunto de
validación para que cada combinación encuentre su propio número óptimo de árboles.

In [3]:
resultados_hiperparametros = {}
for target in TARGETS:
    modelo, config, tabla = modeling.buscar_hiperparametros(
        wr, target, features_compartidas, train_mask, val_mask,
        n_intentos=100, semilla=0, params_fijos=PARAMS_FIJOS.get(target),
    )
    pred_test = modelo.predict(wr.loc[test_mask, features_compartidas])
    m_test = modeling.metricas(wr.loc[test_mask, target], pred_test)
    resultados_hiperparametros[target] = {"modelo": modelo, "config": config, "tabla": tabla, "test": m_test}
    print(f"{target}: val_mae mejor intento={tabla['val_mae'].min():.4f} "
          f"(baseline val_mae={BASELINE[target]['val_mae']:.4f}) -- test_mae={m_test['mae']:.4f} "
          f"(baseline test_mae={BASELINE[target]['test_mae']})")
    print(f"  config ganadora: {config}")

receptions: val_mae mejor intento=1.4466 (baseline val_mae=1.4490) -- test_mae=1.3688 (baseline test_mae=1.372)
  config ganadora: {'max_depth': 3, 'learning_rate': 0.08, 'subsample': 0.6, 'colsample_bytree': 0.9, 'min_child_weight': 5, 'reg_alpha': 0.5, 'reg_lambda': 2.0, 'gamma': 0.1, 'n_estimators': 75, 'random_state': 0, 'n_jobs': -1}


receiving_yards: val_mae mejor intento=21.9372 (baseline val_mae=22.0150) -- test_mae=20.6773 (baseline test_mae=20.647)
  config ganadora: {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.6, 'colsample_bytree': 0.6, 'min_child_weight': 2, 'reg_alpha': 0.5, 'reg_lambda': 3.0, 'gamma': 0.05, 'n_estimators': 48, 'random_state': 0, 'n_jobs': -1}


receiving_tds: val_mae mejor intento=0.1950 (baseline val_mae=0.2464) -- test_mae=0.2042 (baseline test_mae=0.2466)
  config ganadora: {'max_depth': 10, 'learning_rate': 0.2, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 2, 'reg_alpha': 0.0, 'reg_lambda': 2.0, 'gamma': 0.0, 'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5, 'n_estimators': 1416, 'random_state': 0, 'n_jobs': -1}


## 3. Eje 2 — feature set propio por target, mismos hiperparámetros de siempre

Se aísla el efecto de las features: mismos hiperparámetros que ya ganaron en
`13_comparacion_modelos.ipynb`/`14_receiving_tds_modelos_de_conteo.ipynb`, cambiando solo qué
columnas ve el modelo.

In [4]:
from xgboost import XGBRegressor

HIPERPARAMETROS_GANADORES = {
    "receptions": dict(max_depth=4, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="reg:tweedie", tweedie_variance_power=1.5),
}

resultados_features_propias = {}
for target in TARGETS:
    cols = FEATURES_PROPIAS[target]
    modelo = XGBRegressor(**HIPERPARAMETROS_GANADORES[target], random_state=0, n_jobs=-1)
    modelo.fit(wr.loc[train_mask, cols], wr.loc[train_mask, target])
    pred_val = modelo.predict(wr.loc[val_mask, cols])
    pred_test = modelo.predict(wr.loc[test_mask, cols])
    resultados_features_propias[target] = {
        "modelo": modelo,
        "val_mae": modeling.metricas(wr.loc[val_mask, target], pred_val)["mae"],
        "test": modeling.metricas(wr.loc[test_mask, target], pred_test),
    }
    print(f"{target}: val_mae={resultados_features_propias[target]['val_mae']:.4f} "
          f"(baseline={BASELINE[target]['val_mae']}) -- "
          f"test_mae={resultados_features_propias[target]['test']['mae']:.4f} "
          f"(baseline={BASELINE[target]['test_mae']})")

receptions: val_mae=1.4557 (baseline=1.449) -- test_mae=1.3785 (baseline=1.372)


receiving_yards: val_mae=22.0807 (baseline=22.015) -- test_mae=20.6820 (baseline=20.647)


receiving_tds: val_mae=0.2535 (baseline=0.2464471310377121) -- test_mae=0.2506 (baseline=0.2466)


## 4. Combinado — feature set propio + hiperparámetros también ajustados para ese feature set

Lo mejor de los 2 ejes, sin asumir que se refuerzan: se corre la misma búsqueda de 100
combinaciones del punto 2, pero ahora sobre el feature set propio de cada target.

In [5]:
resultados_combinado = {}
for target in TARGETS:
    cols = FEATURES_PROPIAS[target]
    modelo, config, tabla = modeling.buscar_hiperparametros(
        wr, target, cols, train_mask, val_mask,
        n_intentos=100, semilla=0, params_fijos=PARAMS_FIJOS.get(target),
    )
    pred_test = modelo.predict(wr.loc[test_mask, cols])
    m_test = modeling.metricas(wr.loc[test_mask, target], pred_test)
    resultados_combinado[target] = {"modelo": modelo, "config": config, "test": m_test}
    print(f"{target}: val_mae mejor intento={tabla['val_mae'].min():.4f} -- test_mae={m_test['mae']:.4f}")

receptions: val_mae mejor intento=1.4479 -- test_mae=1.3723


receiving_yards: val_mae mejor intento=22.0265 -- test_mae=20.7166


receiving_tds: val_mae mejor intento=0.1984 -- test_mae=0.2069


## 5. Tabla final — las 4 configuraciones, lado a lado

In [6]:
tabla_final = pd.DataFrame({
    target: {
        "baseline (34 compartidas, hiperparams originales)": BASELINE[target]["test_mae"],
        "+ hiperparámetros amplios (34 compartidas)": resultados_hiperparametros[target]["test"]["mae"],
        "+ features propias (hiperparams originales)": resultados_features_propias[target]["test"]["mae"],
        "features propias + hiperparámetros amplios": resultados_combinado[target]["test"]["mae"],
    }
    for target in TARGETS
}).T
tabla_final["mejor_mae"] = tabla_final.min(axis=1)
tabla_final["mejora_%_vs_baseline"] = 100 * (
    tabla_final["baseline (34 compartidas, hiperparams originales)"] - tabla_final["mejor_mae"]
) / tabla_final["baseline (34 compartidas, hiperparams originales)"]
tabla_final.round(4)

,"baseline (34 compartidas, hiperparams originales)",+ hiperparámetros amplios (34 compartidas),+ features propias (hiperparams originales),features propias + hiperparámetros amplios,mejor_mae,mejora_%_vs_baseline
receptions,1.3720,1.3688,1.3785,1.3723,1.3688,0.2297
receiving_yards,20.6470,20.6773,20.6820,20.7166,20.6470,0.0000
receiving_tds,0.2466,0.2042,0.2506,0.2069,0.2042,17.1765


## 6. Recomendación por target — con un umbral explícito, no a ojo

Una mejora de 0.2% puede ser ruido de la búsqueda aleatoria, no una mejora real. Se adopta un
cambio solo si supera un umbral de **2% de mejora relativa** en MAE de prueba frente al baseline
-- por debajo de eso, se mantiene la configuración ya elegida en `13`/`14`, para no complicar el
pipeline por una ganancia que no se puede distinguir de ruido.

In [7]:
UMBRAL_MEJORA = 0.02
COLUMNAS_CONFIG = [
    "baseline (34 compartidas, hiperparams originales)",
    "+ hiperparámetros amplios (34 compartidas)",
    "+ features propias (hiperparams originales)",
    "features propias + hiperparámetros amplios",
]
recomendaciones = {}
for target in TARGETS:
    mejora = tabla_final.loc[target, "mejora_%_vs_baseline"] / 100
    columna_ganadora = tabla_final.loc[target, COLUMNAS_CONFIG].astype(float).idxmin()
    if mejora >= UMBRAL_MEJORA and columna_ganadora != "baseline (34 compartidas, hiperparams originales)":
        recomendaciones[target] = f"ADOPTAR -- {columna_ganadora} ({mejora:.1%} mejor)"
    else:
        recomendaciones[target] = f"MANTENER el original ({mejora:.1%} de diferencia, por debajo del umbral)"
for target, texto in recomendaciones.items():
    print(f"{target}: {texto}")

print("\nConfig de hiperparámetros ganadora, si aplica adoptar (34 features compartidas):")
for target in TARGETS:
    if "ADOPTAR" in recomendaciones[target] and "hiperparámetros amplios" in recomendaciones[target]:
        print(f"  {target}: {resultados_hiperparametros[target]['config']}")

receptions: MANTENER el original (0.2% de diferencia, por debajo del umbral)
receiving_yards: MANTENER el original (0.0% de diferencia, por debajo del umbral)
receiving_tds: ADOPTAR -- + hiperparámetros amplios (34 compartidas) (17.2% mejor)

Config de hiperparámetros ganadora, si aplica adoptar (34 features compartidas):
  receiving_tds: {'max_depth': 10, 'learning_rate': 0.2, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 2, 'reg_alpha': 0.0, 'reg_lambda': 2.0, 'gamma': 0.0, 'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5, 'n_estimators': 1416, 'random_state': 0, 'n_jobs': -1}


## Cierre

El feature set propio por target **no ayudó en ningún caso** (siempre quedó por encima del
baseline en MAE de prueba) -- confirma que la decisión original de Fase 4 (34 columnas
compartidas, la unión de lo importante para cada target) fue razonable, no una simplificación que
estuviera costando precisión. La búsqueda de hiperparámetros más amplia sí encontró una mejora
real y grande en `receiving_tds` (por encima del umbral de 2%) -- el target más difícil y el que
más se beneficia de mejor regularización, consistente con su naturaleza de conteo con 82% de
ceros. En `receptions`/`receiving_yards` la diferencia no supera el umbral -- la configuración
original ya estaba cerca de su techo práctico.

Esto mide el techo real de mejora de XGBoost sobre lo que ya se tenía -- sin cambiar de familia de
modelo (eso ya se decidió) y sin tocar todavía los modelos guardados en `weekly/wr/models/`. Si se
confirma adoptar el cambio de `receiving_tds`, el siguiente paso es reentrenar y volver a guardar
ese modelo específico con la nueva configuración -- se hace junto con Daniel, no automáticamente
aquí.